# Parte 1 — Datos y Tokenización
### Workshop: Clasificación de Emociones en Twitter

Este notebook cubre:
- Exploración del dataset TweetEval Emotion (EDA)
- Análisis comparativo de tokenización: DistilBERT vs BERTweet
- Guarda el dataset crudo para que los siguientes notebooks lo carguen directamente

> **Referencia:** Barbieri et al. (2020). *TweetEval: Unified Benchmark and Comparative Evaluation for Tweet Classification*. EMNLP Findings.

In [ ]:
# !pip install 'transformers[torch]' 'accelerate>=1.1.0' datasets evaluate scikit-learn matplotlib seaborn -q

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

SEED        = 42
MAX_LENGTH  = 128
LABEL_NAMES = ["anger", "joy", "optimism", "sadness"]
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
PALETTE     = {"anger": "#e74c3c", "joy": "#f1c40f",
               "optimism": "#2ecc71", "sadness": "#3498db"}

np.random.seed(SEED)
torch.manual_seed(SEED)
print("Config OK")

## 1. Carga del dataset

In [ ]:
raw = load_dataset("tweet_eval", "emotion")
print(raw)
print("\nEjemplos por split:")
for split in raw:
    print(f"  {split}: {len(raw[split])} ejemplos")

In [ ]:
print("Primeros 6 ejemplos del split de entrenamiento:\n")
for ex in raw["train"].select(range(6)):
    print(f"  [{ID2LABEL[ex['label']].upper():<10}]  {ex['text']}")

## 2. EDA

In [ ]:
df = raw["train"].to_pandas()
df["emotion"]    = df["label"].map(ID2LABEL)
df["n_chars"]    = df["text"].str.len()
df["n_words"]    = df["text"].str.split().str.len()
df["n_hashtags"] = df["text"].str.count(r"#\w+")
df["n_mentions"] = df["text"].str.count(r"@\w+")

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
counts = df["emotion"].value_counts().reindex(LABEL_NAMES)
bars = ax1.bar(counts.index, counts.values,
               color=[PALETTE[k] for k in counts.index], edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val}", ha="center", fontweight="bold", fontsize=9)
ax1.set_title("Distribución de clases (train)", fontweight="bold")
ax1.set_ylabel("Ejemplos")
ax1.spines[["top", "right"]].set_visible(False)

ax2 = fig.add_subplot(gs[0, 1:3])
for emotion in LABEL_NAMES:
    subset = df[df["emotion"] == emotion]
    ax2.hist(subset["n_words"], bins=30, alpha=0.55,
             label=emotion, color=PALETTE[emotion], edgecolor="none")
ax2.set_title("Distribución de longitud por emoción (palabras)", fontweight="bold")
ax2.set_xlabel("Palabras"); ax2.set_ylabel("Frecuencia")
ax2.legend(); ax2.spines[["top", "right"]].set_visible(False)

ax3 = fig.add_subplot(gs[1, 0])
means = df.groupby("emotion")[["n_hashtags", "n_mentions"]].mean().reindex(LABEL_NAMES)
x = np.arange(len(LABEL_NAMES)); w = 0.35
ax3.bar(x - w/2, means["n_hashtags"], w, label="#hashtags", color="#8e44ad", edgecolor="white")
ax3.bar(x + w/2, means["n_mentions"], w, label="@mentions", color="#16a085", edgecolor="white")
ax3.set_xticks(x); ax3.set_xticklabels(LABEL_NAMES)
ax3.set_title("Media de #hashtags y @mentions", fontweight="bold")
ax3.legend(); ax3.spines[["top", "right"]].set_visible(False)

ax4 = fig.add_subplot(gs[1, 1:3])
df.boxplot(column="n_words", by="emotion",
           positions=range(len(LABEL_NAMES)), ax=ax4,
           patch_artist=True, boxprops=dict(facecolor="#ecf0f1"),
           medianprops=dict(color="red", linewidth=2))
ax4.set_xticklabels(LABEL_NAMES)
ax4.set_title("Longitud por emoción", fontweight="bold")
ax4.set_xlabel(""); ax4.set_ylabel("Palabras")
plt.sca(ax4); plt.title("")

fig.suptitle("TweetEval Emotion — EDA (split train)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print("\nConteo de clases en cada split:")
for split in raw:
    c = Counter(raw[split]["label"])
    total = sum(c.values())
    print(f"  {split}: " + "  ".join(
        f"{ID2LABEL[k]}={v} ({v/total*100:.0f}%)" for k, v in sorted(c.items())))

### Observaciones del EDA

Antes de continuar, anota tus observaciones:

- ¿Hay desequilibrio de clases? ¿Cuál es la clase mayoritaria?
- ¿Cuántos tokens tienen aproximadamente los tweets? ¿Justifica `MAX_LENGTH=128`?
- ¿Qué características de Twitter (hashtags, mentions) podrían afectar a un tokenizador entrenado en texto formal?

> ⚠️ Con clases desbalanceadas la *accuracy* puede ser engañosa. Usaremos **F1 macro** como métrica principal.

## 3. Tokenización comparativa

DistilBERT fue pre-entrenado en Wikipedia y BookCorpus — texto formal. Su tokenizador no conoce abreviaciones de Twitter, hashtags ni emojis.

BERTweet fue pre-entrenado en 850M tweets. Su tokenizador fue entrenado sobre texto de Twitter, incluyendo `@mentions`, `#hashtags` y emojis normalizados.

In [ ]:
tok_distilbert = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tok_bertweet   = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True)

def compare_tokenization(text, label=""):
    toks_d = tok_distilbert.tokenize(text)
    toks_b = tok_bertweet.tokenize(text)
    print(f"\n{'─'*65}")
    print(f"  [{label}] {text}" if label else f"  {text}")
    print(f"{'─'*65}")
    print(f"  DistilBERT  ({len(toks_d):2d} tokens): {toks_d}")
    print(f"  BERTweet    ({len(toks_b):2d} tokens): {toks_b}")

examples = [
    ("I'm so happy today!!! 😊",                        "joy"),
    ("this is absolutely disgusting #angry #wtf",       "anger"),
    ("@user can't believe what just happened... 😢",    "sadness"),
    ("RT @user: The future looks bright #hope #goals",  "optimism"),
]

for text, label in examples:
    compare_tokenization(text, label)

### 📝 TODO 2.1 — Análisis de tokenización

1. ¿Cuántos tokens produce en promedio cada tokenizador sobre todos los tweets del split de train?
2. ¿Cómo tokeniza DistilBERT el hashtag `#wtf` comparado con BERTweet?
3. ¿Qué pasa con `@user`? ¿Y con emojis como `😊`?
4. Busca 2 tweets del dataset (`raw["train"][i]["text"]`) que creas que mostrarán diferencias interesantes y analízalos.

In [ ]:
# TODO 2.1 ── Análisis de tokenización
# ─────────────────────────────────────────────────────────────────────────────
# 1. Calcula el promedio de tokens para todos los tweets del split de train
#    con cada tokenizador. ¿Cuál produce secuencias más largas?

# YOUR CODE HERE


# 2. Encuentra un tweet del dataset que tenga emojis o hashtags
#    y muestra su tokenización comparativa

# YOUR CODE HERE


# 3. ¿Qué porcentaje de tweets superarían MAX_LENGTH=128 con cada tokenizador?

# YOUR CODE HERE

<details>
<summary>💡 Pista</summary>

Para el punto 1, itera sobre `raw["train"]["text"]` y llama `tokenizer(text)["input_ids"]` en cada tweet. Recuerda que `len(input_ids)` incluye los tokens especiales `[CLS]` y `[SEP]`.

</details>